# Memory slice: HAR and naive

HAR is the benchmark every other model has to beat.


## 1. Setup and data

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import TimeSeriesSplit


if os.path.basename(os.getcwd()) == "memory":
    os.chdir("../..")
sys.path.append(os.getcwd())
from src.foundation.evaluate import evaluate, save_predictions

RUN_TEST = False   # switch to True only on the group's test-run call

In [ ]:
# Load the shared model table 
model_table = pd.read_csv("data/processed/model_table.csv", parse_dates=["date"])
train = model_table[model_table["split"] == "train"].reset_index(drop=True)
test = model_table[model_table["split"] == "test"].reset_index(drop=True)

HAR_COLS = ["f1", "f2", "f3"]   # volatility over today, the last week, the last month
y_train = train["y"]

# The same CV as every other model: 5 folds inside the training years, 5-row gap
tscv = TimeSeriesSplit(n_splits=5, gap=5)
print(f"Training rows: {len(train)} ({train['date'].min().date()} to {train['date'].max().date()})")

## 2. HAR model

HAR (Corsi, 2009) is a linear regression of next week's log volatility on three past volatilities:

y = b0 + b1·f1 + b2·f2 + b3·f3

f1 is today's move, f2 the last week and f3 the last month. 

There is nothing to tune, so we fit it once on all training rows. We use statsmodels instead of scikit-learn because it prints the coefficients with their standard errors.

In [ ]:
X_train = sm.add_constant(train[HAR_COLS])
# Next week's targets overlap by 4 days, so neighbouring errors are correlated.
# HAC standard errors (lag 4) allow for that; the coefficients themselves don't change.
har = sm.OLS(y_train, X_train).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(har.summary().tables[1])

### 2.1 Do the coefficients make sense?

Corsi's idea is that more past volatility means more future volatility, so all three slopes should be positive. Their sum tells us how persistent volatility is: close to 1 means a shock fades slowly.

In [ ]:
slopes = har.params[HAR_COLS]
print("All slopes positive:", (slopes > 0).all())
print(f"Sum of slopes: {slopes.sum():.3f}")

**Result.** All three slopes are positive and they add up to 0.83, so volatility is persistent but a shock slowly fades, as Corsi (2009) describes. Last month (f3, 0.56) carries the most weight, then last week (f2, 0.26). Today's move (f1) adds almost nothing (0.007, p = 0.38): one day's return is very noisy, and the weekly and monthly volatility already contain it.

## 3. HAR on each CV fold

For each fold we fit HAR on the earlier years and score it on the next block with the shared `evaluate()`.

In [ ]:
def cv_scores_har():
    """Fit HAR on each fold's earlier years, score on the next block. One row per fold."""
    rows = []
    for fit_idx, val_idx in tscv.split(train):
        fold_model = sm.OLS(y_train[fit_idx], sm.add_constant(train.loc[fit_idx, HAR_COLS])).fit()
        y_pred = fold_model.predict(sm.add_constant(train.loc[val_idx, HAR_COLS]))
        rows.append(evaluate(y_train[val_idx], y_pred))
    return pd.DataFrame(rows, index=range(1, 6)).rename_axis("fold")

har_scores = cv_scores_har()
print("Mean over the 5 folds:", har_scores.mean().round(3).to_dict())
har_scores.round(3)

**Result.** On average HAR scores RMSE 0.508 and R² 0.18. It does best in fold 2, which covers the 2008 crisis (R² 0.42), and worst in fold 5 (May 2013 – Dec 2014, a calm period), where R² is below zero (−0.14). For comparison, Victor's Elastic Net scores a CV RMSE of 0.437 and Sadman's LightGBM 0.529 (from their notebooks), on the same folds.

## 4. Naive forecast on each CV fold

The simplest forecast: next week's volatility = last week's (f2). There is nothing to fit, so we just score f2 on each fold's block. Any model that can't beat this isn't learning anything useful.

In [ ]:
naive_scores = pd.DataFrame([evaluate(y_train[val_idx], train.loc[val_idx, "f2"])
                             for _, val_idx in tscv.split(train)], index=range(1, 6)).rename_axis("fold")
print("Mean over the 5 folds:", naive_scores.mean().round(3).to_dict())
naive_scores.round(3)

### 4.1 Naive vs HAR, fold by fold

In [ ]:
comparison = pd.DataFrame({"naive RMSE": naive_scores["RMSE"], "HAR RMSE": har_scores["RMSE"],
                           "naive R2": naive_scores["R2"], "HAR R2": har_scores["R2"]})
comparison["HAR better"] = comparison["HAR RMSE"] < comparison["naive RMSE"]
comparison.round(3)

**Result.** Naive averages RMSE 0.566 and R² −0.03, so on average it is no better than predicting the mean, and its R² is below zero in 3 of 5 folds. HAR beats it in 4 of 5 folds (mean RMSE 0.508). The exception is fold 2, the 2008 crisis: naive is slightly better there (0.480 vs 0.487), because last week's volatility reacts straight away when the market suddenly gets wilder, while HAR still gives weight to the calmer last month (f3). These naive numbers match the first CV score in the main notebook (Step 9).

## 5. Test run 

Fit HAR once on all training rows, forecast the test period, and save `results/preds_har.csv` and `results/preds_naive.csv` (naive is just f2). It refuses to run if the file already exists, so the test set is used only once. Nothing above this cell changes after it has been run.

In [ ]:
if RUN_TEST:
    for f in ["results/preds_har.csv", "results/preds_naive.csv"]:
        assert not os.path.exists(f), f"{f} exists: test set already used"

    har_pred = har.predict(sm.add_constant(test[HAR_COLS]))
    save_predictions("har", test, har_pred)
    save_predictions("naive", test, test["f2"])
    print("Saved results/preds_har.csv and results/preds_naive.csv")
    print("HAR test scores:  ", {k: round(float(v), 3) for k, v in evaluate(test["y"], har_pred).items()})
    print("Naive test scores:", {k: round(float(v), 3) for k, v in evaluate(test["y"], test["f2"]).items()})
else:
    print("Locked: RUN_TEST is False")